# 03 — Model (BitNet b1.58)

BitNet Transformer with ternary weights (-1, 0, +1).
Saves to **Google Drive**.

> Runs on **Google Colab** (free T4 tier).

## 1. Install dependencies

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers tokenizers

## 2. Mount Google Drive (MANDATORY)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Imports & Config

In [ ]:
import math, torch, json, os
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

DRIVE_BASE = "/content/drive/MyDrive/ANTHOR"
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

## 4. Configuration

In [ ]:
@dataclass
class BitNetConfig:
    vocab_size: int = 16384
    n_layers: int = 12
    n_heads: int = 12
    n_kv_heads: int = 4
    d_model: int = 768
    d_ff: int = 2048
    max_seq_len: int = 1024
    dropout: float = 0.1
    use_bitnet: bool = True
    rope_theta: float = 10000.0
    
    def __post_init__(self):
        self.head_dim = self.d_model // self.n_heads
        assert self.d_model % self.n_heads == 0
        assert self.n_heads % self.n_kv_heads == 0

config = BitNetConfig(
    vocab_size=16384,
    n_layers=12,
    n_heads=12,
    n_kv_heads=4,
    d_model=768,
    d_ff=2048,
    max_seq_len=1024,
)
print(config)

## 5. RMSNorm

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return self.weight * (x / rms)

## 6. RoPE

In [ ]:
def precompute_freqs_cis(dim, max_seq_len, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(freqs), freqs)

def apply_rotary_emb(x, freqs_cis):
    x_ = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    freqs_cis = freqs_cis.unsqueeze(0).unsqueeze(2)
    return torch.view_as_real(x_ * freqs_cis).flatten(-2).type_as(x)

## 7. BitLinear (Ternary Quantization)

In [ ]:
def quantize_weights_ternary(w):
    gamma = w.abs().mean()
    w_scaled = w / (gamma + 1e-5)
    w_q = torch.sign(w_scaled) * (w_scaled.abs() > 0.5).float()
    return w_q, gamma

class BitLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=False):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02)
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
    
    def forward(self, x):
        w_q, gamma = quantize_weights_ternary(self.weight)
        w = w_q * gamma + (self.weight - self.weight.detach())
        return F.linear(x, w, self.bias)

## 8. Grouped-Query Attention

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads = config.n_heads
        self.n_kv_heads = config.n_kv_heads
        self.head_dim = config.head_dim
        self.n_rep = self.n_heads // self.n_kv_heads
        
        Linear = BitLinear if config.use_bitnet else nn.Linear
        self.wq = Linear(config.d_model, self.n_heads * self.head_dim, bias=False)
        self.wk = Linear(config.d_model, self.n_kv_heads * self.head_dim, bias=False)
        self.wv = Linear(config.d_model, self.n_kv_heads * self.head_dim, bias=False)
        self.wo = Linear(self.n_heads * self.head_dim, config.d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout)
    
    def forward(self, x, freqs_cis, mask=None):
        batch, seq_len, _ = x.shape
        
        q = self.wq(x).view(batch, seq_len, self.n_heads, self.head_dim)
        k = self.wk(x).view(batch, seq_len, self.n_kv_heads, self.head_dim)
        v = self.wv(x).view(batch, seq_len, self.n_kv_heads, self.head_dim)
        
        q = apply_rotary_emb(q, freqs_cis[:seq_len])
        k = apply_rotary_emb(k, freqs_cis[:seq_len])
        
        k = k.repeat_interleave(self.n_rep, dim=2)
        v = v.repeat_interleave(self.n_rep, dim=2)
        
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores + mask
        
        attn = self.dropout(F.softmax(scores, dim=-1))
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(batch, seq_len, -1)
        return self.wo(out)

## 9. SwiGLU FFN

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()
        Linear = BitLinear if config.use_bitnet else nn.Linear
        self.w1 = Linear(config.d_model, config.d_ff, bias=False)
        self.w2 = Linear(config.d_ff, config.d_model, bias=False)
        self.w3 = Linear(config.d_model, config.d_ff, bias=False)
        self.dropout = nn.Dropout(config.dropout)
    
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

## 10. Transformer Block

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn = GroupedQueryAttention(config)
        self.ffn = SwiGLU(config)
        self.attn_norm = RMSNorm(config.d_model)
        self.ffn_norm = RMSNorm(config.d_model)
    
    def forward(self, x, freqs_cis, mask=None):
        x = x + self.attn(self.attn_norm(x), freqs_cis, mask)
        x = x + self.ffn(self.ffn_norm(x))
        return x

## 11. Full BitNet Model

In [ ]:
class BitNetTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.tok_embeddings = nn.Embedding(config.vocab_size, config.d_model)
        self.layers = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        self.norm = RMSNorm(config.d_model)
        self.output = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.output.weight = self.tok_embeddings.weight
        
        self.register_buffer("freqs_cis", precompute_freqs_cis(config.head_dim, config.max_seq_len, config.rope_theta))
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, input_ids, targets=None):
        batch, seq_len = input_ids.shape
        x = self.tok_embeddings(input_ids)
        mask = torch.triu(torch.full((seq_len, seq_len), float("-inf")), diagonal=1)
        
        for layer in self.layers:
            x = layer(x, self.freqs_cis[:seq_len], mask)
        
        x = self.norm(x)
        logits = self.output(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        
        return logits, loss
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters())

## 12. Test & Save to Drive

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = BitNetTransformer(config).to(device)
n_params = model.count_parameters()

print(f"Model: {n_params / 1e6:.2f}M parameters")

# Test forward pass
dummy = torch.randint(0, config.vocab_size, (2, 128)).to(device)
with torch.no_grad():
    logits, loss = model(dummy, dummy)
print(f"Forward pass OK: loss={loss.item():.4f}")

# ── Save to Google Drive (MANDATORY) ──
config_path = f"{DRIVE_BASE}/model_config.json"
ckpt_path = f"{DRIVE_BASE}/model_init.pt"

config_dict = {"vocab_size": config.vocab_size, "n_layers": config.n_layers,
               "n_heads": config.n_heads, "n_kv_heads": config.n_kv_heads,
               "d_model": config.d_model, "d_ff": config.d_ff,
               "max_seq_len": config.max_seq_len, "dropout": config.dropout,
               "use_bitnet": config.use_bitnet, "rope_theta": config.rope_theta}

with open(config_path, "w") as f:
    json.dump(config_dict, f, indent=2)
print(f"Saved config to: {config_path}")

torch.save({"model_state_dict": model.state_dict(), "config": config_dict, "n_params": n_params}, ckpt_path)
print(f"Saved model to: {ckpt_path}")

---

✅ **Next:** `04_train.ipynb` (loads from Google Drive)

Saved to Drive:
- `model_config.json`
- `model_init.pt`